In [1]:
#Install/import libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
#Create a sample dataset
np.random.seed(42)

data = {
    "temperature": np.random.uniform(20, 40, 200),
    "humidity": np.random.uniform(40, 90, 200),
    "rainfall": np.random.uniform(0, 200, 200),
    "soil_moisture": np.random.uniform(20, 80, 200),
    "irrigation": np.random.uniform(10, 100, 200)
}

df = pd.DataFrame(data)

# Simulated crop yield
df["yield"] = (
    0.05 * df["soil_moisture"]
    + 0.03 * df["rainfall"]
    + 0.08 * df["irrigation"]
    - 0.02 * (df["temperature"] - 30) ** 2
    + np.random.normal(0, 1, 200)
)

print(df.head())

   temperature   humidity    rainfall  soil_moisture  irrigation      yield
0    27.490802  72.101582   20.624774      30.136104   73.651477   8.069446
1    39.014286  44.206998  180.510581      36.715420   23.728514   6.188878
2    34.639879  48.081436  101.050474      30.620629   61.865952   9.461450
3    31.973170  84.927709  165.291493      25.322152   64.604354  11.925919
4    23.120373  70.321453   64.009920      27.238152   48.171760   6.749151


In [3]:
#Separate input and output
#Our ML model needs:

#X = agricultural conditions

#y = crop yield


X = df[
    [
        "temperature",
        "humidity",
        "rainfall",
        "soil_moisture",
        "irrigation"
    ]
]

y = df["yield"]

In [ ]:
#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Here:

#80% → training
#20% → testing

In [5]:
#Train the yield prediction model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
#Test the model

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("R2 Score:", r2)

#If the model works well, we'll see an R² value reasonably close to 1.

Mean Absolute Error: 1.0996292856143772
R2 Score: 0.78912557760712


In [9]:
#Make a yield prediction
#Suppose the farm currently has:

#Temperature = 30°C
#Humidity = 70%
#Rainfall = 50 mm
#Soil moisture = 50%
#Irrigation = 60

#We can predict the yield:


new_data = pd.DataFrame({
    "temperature": [30],
    "humidity": [70],
    "rainfall": [50],
    "soil_moisture": [50],
    "irrigation": [60]
})

predicted_yield = model.predict(new_data)

print("Predicted Yield:", predicted_yield[0])

Predicted Yield: 8.641137526381504


In [11]:
#Grey Wolf Optimization 🐺

#This is where OptiCrop becomes different from a normal ML project.

#Instead of saying:

#"Irrigate with 60 units."

#we ask:

#"What irrigation amount should we use to maximize crop yield while minimizing water consumption?"

#First, let's create a simple objective function.

def fitness(irrigation):
    
    test_data = pd.DataFrame({
        "temperature": [30],
        "humidity": [70],
        "rainfall": [50],
        "soil_moisture": [50],
        "irrigation": [irrigation]
    })
    
    predicted_yield = model.predict(test_data)[0]
    
    # We want high yield and low water usage
    fitness_value = -predicted_yield + 0.01 * irrigation
    
    return fitness_value

    #The optimizer will try to minimize this fitness value.

In [12]:
#Implement Grey Wolf Optimization

def GWO(objective_function, lower_bound, upper_bound,
        n_wolves=10, max_iterations=30):

    # Initialize wolves
    wolves = np.random.uniform(
        lower_bound,
        upper_bound,
        n_wolves
    )

    # Alpha, Beta and Delta wolves
    alpha = None
    beta = None
    delta = None

    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    for iteration in range(max_iterations):

        # Evaluate every wolf
        for i in range(n_wolves):

            score = objective_function(wolves[i])

            # Alpha
            if score < alpha_score:
                delta_score = beta_score
                delta = beta

                beta_score = alpha_score
                beta = alpha

                alpha_score = score
                alpha = wolves[i]

            # Beta
            elif score < beta_score:
                delta_score = beta_score
                delta = beta

                beta_score = score
                beta = wolves[i]

            # Delta
            elif score < delta_score:
                delta_score = score
                delta = wolves[i]

        # GWO coefficient
        a = 2 - iteration * (2 / max_iterations)

        # Update wolf positions
        for i in range(n_wolves):

            r1 = np.random.random()
            r2 = np.random.random()

            A1 = 2 * a * r1 - a
            C1 = 2 * r2

            D_alpha = abs(C1 * alpha - wolves[i])
            X1 = alpha - A1 * D_alpha

            r1 = np.random.random()
            r2 = np.random.random()

            A2 = 2 * a * r1 - a
            C2 = 2 * r2

            D_beta = abs(C2 * beta - wolves[i])
            X2 = beta - A2 * D_beta

            r1 = np.random.random()
            r2 = np.random.random()

            A3 = 2 * a * r1 - a
            C3 = 2 * r2

            D_delta = abs(C3 * delta - wolves[i])
            X3 = delta - A3 * D_delta

            # Update position
            wolves[i] = (X1 + X2 + X3) / 3

            # Keep within boundaries
            wolves[i] = np.clip(
                wolves[i],
                lower_bound,
                upper_bound
            )

    return alpha, alpha_score

In [13]:
#Run OptiCrop
best_irrigation, best_fitness = GWO(
    fitness,
    lower_bound=10,
    upper_bound=100,
    n_wolves=10,
    max_iterations=30
)

print("Optimal Irrigation:", best_irrigation)
print("Fitness:", best_fitness)


Optimal Irrigation: 89.46123858739378
Fitness: -8.798497539487116


In [14]:
optimal_data = pd.DataFrame({
    "temperature": [30],
    "humidity": [70],
    "rainfall": [50],
    "soil_moisture": [50],
    "irrigation": [best_irrigation]
})

optimal_yield = model.predict(optimal_data)[0]

print("Optimal Irrigation:", best_irrigation)
print("Predicted Yield:", optimal_yield)

Optimal Irrigation: 89.46123858739378
Predicted Yield: 9.693109925361053


In [15]:
#Let's build our GUI  using Tkinter

import tkinter as tk
from tkinter import ttk, messagebox

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


# ============================================================
# 1. CREATE SAMPLE AGRICULTURAL DATA
# ============================================================

np.random.seed(42)

data = {
    "temperature": np.random.uniform(20, 40, 200),
    "humidity": np.random.uniform(40, 90, 200),
    "rainfall": np.random.uniform(0, 200, 200),
    "soil_moisture": np.random.uniform(20, 80, 200),
    "irrigation": np.random.uniform(10, 100, 200)
}

df = pd.DataFrame(data)

# Simulated crop yield
df["yield"] = (
    0.05 * df["soil_moisture"]
    + 0.03 * df["rainfall"]
    + 0.08 * df["irrigation"]
    - 0.02 * (df["temperature"] - 30) ** 2
    + np.random.normal(0, 1, 200)
)


# ============================================================
# 2. TRAIN MACHINE LEARNING MODEL
# ============================================================

X = df[
    [
        "temperature",
        "humidity",
        "rainfall",
        "soil_moisture",
        "irrigation"
    ]
]

y = df["yield"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)


# ============================================================
# 3. GREY WOLF OPTIMIZATION
# ============================================================

def GWO(objective_function,
        lower_bound,
        upper_bound,
        n_wolves=10,
        max_iterations=30):

    wolves = np.random.uniform(
        lower_bound,
        upper_bound,
        n_wolves
    )

    alpha = None
    beta = None
    delta = None

    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    for iteration in range(max_iterations):

        for i in range(n_wolves):

            score = objective_function(wolves[i])

            # Alpha
            if score < alpha_score:

                delta_score = beta_score
                delta = beta

                beta_score = alpha_score
                beta = alpha

                alpha_score = score
                alpha = wolves[i]

            # Beta
            elif score < beta_score:

                delta_score = beta_score
                delta = beta

                beta_score = score
                beta = wolves[i]

            # Delta
            elif score < delta_score:

                delta_score = score
                delta = wolves[i]

        a = 2 - iteration * (2 / max_iterations)

        for i in range(n_wolves):

            # Alpha
            r1 = np.random.random()
            r2 = np.random.random()

            A1 = 2 * a * r1 - a
            C1 = 2 * r2

            D_alpha = abs(C1 * alpha - wolves[i])
            X1 = alpha - A1 * D_alpha

            # Beta
            r1 = np.random.random()
            r2 = np.random.random()

            A2 = 2 * a * r1 - a
            C2 = 2 * r2

            D_beta = abs(C2 * beta - wolves[i])
            X2 = beta - A2 * D_beta

            # Delta
            r1 = np.random.random()
            r2 = np.random.random()

            A3 = 2 * a * r1 - a
            C3 = 2 * r2

            D_delta = abs(C3 * delta - wolves[i])
            X3 = delta - A3 * D_delta

            # Update position
            wolves[i] = (X1 + X2 + X3) / 3

            # Boundary control
            wolves[i] = np.clip(
                wolves[i],
                lower_bound,
                upper_bound
            )

    return alpha


# ============================================================
# 4. GUI FUNCTIONS
# ============================================================

def get_inputs():

    try:

        temperature = float(temp_entry.get())
        humidity = float(humidity_entry.get())
        rainfall = float(rainfall_entry.get())
        soil_moisture = float(soil_entry.get())
        irrigation = float(irrigation_entry.get())

        return (
            temperature,
            humidity,
            rainfall,
            soil_moisture,
            irrigation
        )

    except ValueError:

        messagebox.showerror(
            "Invalid Input",
            "Please enter valid numerical values."
        )

        return None


def predict_yield():

    values = get_inputs()

    if values is None:
        return

    temperature, humidity, rainfall, soil_moisture, irrigation = values

    input_data = pd.DataFrame({
        "temperature": [temperature],
        "humidity": [humidity],
        "rainfall": [rainfall],
        "soil_moisture": [soil_moisture],
        "irrigation": [irrigation]
    })

    prediction = model.predict(input_data)[0]

    yield_result.config(
        text=f"{prediction:.2f} tons"
    )


def optimize_irrigation():

    values = get_inputs()

    if values is None:
        return

    temperature, humidity, rainfall, soil_moisture, current_irrigation = values

    # Objective function
    def fitness(irrigation):

        test_data = pd.DataFrame({
            "temperature": [temperature],
            "humidity": [humidity],
            "rainfall": [rainfall],
            "soil_moisture": [soil_moisture],
            "irrigation": [irrigation]
        })

        predicted_yield = model.predict(test_data)[0]

        # Minimize fitness
        # High yield = good
        # Low water consumption = good
        fitness_value = (
            -predicted_yield
            + 0.01 * irrigation
        )

        return fitness_value

    # Run GWO
    optimal_irrigation = GWO(
        fitness,
        lower_bound=10,
        upper_bound=100,
        n_wolves=10,
        max_iterations=30
    )

    # Predict yield using optimized irrigation
    optimal_data = pd.DataFrame({
        "temperature": [temperature],
        "humidity": [humidity],
        "rainfall": [rainfall],
        "soil_moisture": [soil_moisture],
        "irrigation": [optimal_irrigation]
    })

    optimal_yield = model.predict(optimal_data)[0]

    irrigation_result.config(
        text=f"{optimal_irrigation:.2f}"
    )

    optimized_yield_result.config(
        text=f"{optimal_yield:.2f} tons"
    )


def clear_fields():

    temp_entry.delete(0, tk.END)
    humidity_entry.delete(0, tk.END)
    rainfall_entry.delete(0, tk.END)
    soil_entry.delete(0, tk.END)
    irrigation_entry.delete(0, tk.END)

    yield_result.config(text="--")
    irrigation_result.config(text="--")
    optimized_yield_result.config(text="--")


# ============================================================
# 5. CREATE TKINTER WINDOW
# ============================================================

root = tk.Tk()

root.title("OptiCrop - Smart Irrigation & Yield Forecasting")

root.geometry("900x650")

root.resizable(False, False)


# ============================================================
# 6. HEADER
# ============================================================

header = tk.Frame(
    root,
    padx=20,
    pady=20
)

header.pack(fill="x")

title = tk.Label(
    header,
    text="🌱 OptiCrop",
    font=("Arial", 28, "bold")
)

title.pack()

subtitle = tk.Label(
    header,
    text="Grey Wolf–Optimized Irrigation & Yield Forecasting",
    font=("Arial", 13)
)

subtitle.pack(pady=5)


# ============================================================
# 7. INPUT FRAME
# ============================================================

input_frame = ttk.LabelFrame(
    root,
    text="Agricultural Parameters",
    padding=20
)

input_frame.pack(
    padx=30,
    pady=10,
    fill="x"
)


# Temperature
ttk.Label(
    input_frame,
    text="Temperature (°C)"
).grid(row=0, column=0, padx=10, pady=10)

temp_entry = ttk.Entry(
    input_frame,
    width=20
)

temp_entry.grid(row=0, column=1, padx=10)


# Humidity
ttk.Label(
    input_frame,
    text="Humidity (%)"
).grid(row=0, column=2, padx=10)

humidity_entry = ttk.Entry(
    input_frame,
    width=20
)

humidity_entry.grid(row=0, column=3, padx=10)


# Rainfall
ttk.Label(
    input_frame,
    text="Rainfall (mm)"
).grid(row=1, column=0, padx=10, pady=10)

rainfall_entry = ttk.Entry(
    input_frame,
    width=20
)

rainfall_entry.grid(row=1, column=1, padx=10)


# Soil Moisture
ttk.Label(
    input_frame,
    text="Soil Moisture (%)"
).grid(row=1, column=2, padx=10)

soil_entry = ttk.Entry(
    input_frame,
    width=20
)

soil_entry.grid(row=1, column=3, padx=10)


# Irrigation
ttk.Label(
    input_frame,
    text="Current Irrigation"
).grid(row=2, column=0, padx=10, pady=10)

irrigation_entry = ttk.Entry(
    input_frame,
    width=20
)

irrigation_entry.grid(row=2, column=1, padx=10)


# ============================================================
# 8. BUTTONS
# ============================================================

button_frame = tk.Frame(root)

button_frame.pack(pady=20)


predict_button = ttk.Button(
    button_frame,
    text="Predict Yield",
    command=predict_yield
)

predict_button.grid(
    row=0,
    column=0,
    padx=10
)


optimize_button = ttk.Button(
    button_frame,
    text="🐺 Optimize Irrigation",
    command=optimize_irrigation
)

optimize_button.grid(
    row=0,
    column=1,
    padx=10
)


clear_button = ttk.Button(
    button_frame,
    text="Clear",
    command=clear_fields
)

clear_button.grid(
    row=0,
    column=2,
    padx=10
)


# ============================================================
# 9. RESULTS
# ============================================================

result_frame = ttk.LabelFrame(
    root,
    text="OptiCrop Results",
    padding=25
)

result_frame.pack(
    padx=30,
    pady=10,
    fill="x"
)


# Current predicted yield
ttk.Label(
    result_frame,
    text="Predicted Yield:"
).grid(
    row=0,
    column=0,
    padx=20,
    pady=15
)

yield_result = ttk.Label(
    result_frame,
    text="--",
    font=("Arial", 14, "bold")
)

yield_result.grid(
    row=0,
    column=1
)


# Optimal irrigation
ttk.Label(
    result_frame,
    text="Optimal Irrigation:"
).grid(
    row=1,
    column=0,
    padx=20,
    pady=15
)

irrigation_result = ttk.Label(
    result_frame,
    text="--",
    font=("Arial", 14, "bold")
)

irrigation_result.grid(
    row=1,
    column=1
)


# Optimized yield
ttk.Label(
    result_frame,
    text="Yield After Optimization:"
).grid(
    row=2,
    column=0,
    padx=20,
    pady=15
)

optimized_yield_result = ttk.Label(
    result_frame,
    text="--",
    font=("Arial", 14, "bold")
)

optimized_yield_result.grid(
    row=2,
    column=1
)


# ============================================================
# 10. FOOTER
# ============================================================

footer = tk.Label(
    root,
    text="OptiCrop | AI/ML + Grey Wolf Optimization",
    font=("Arial", 10)
)

footer.pack(
    side="bottom",
    pady=15
)


# ============================================================
# 11. START APPLICATION
# ============================================================

root.mainloop()